In [ ]:
['start_xy', 'end_xy', 'start_yaw', 'end_yaw', 'start_speed', 'end_speed', 'avg_speed', 'direction', 'turn_label', 'lane_change']

In [2]:
from __future__ import annotations

import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from tqdm import tqdm
import os

IMG_ROOT = Path('/zfsauton/scratch/eshau/imgs')
INSTRUCTION_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/annotations/')
TARGET_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/manual_instruction/')
os.makedirs(TARGET_ROOT, exist_ok=True)


def write_jsonl_with_byte_index(records, output_path: Path):
    byte_offsets = []
    with output_path.open('wb') as out_f:
        for record in records:
            byte_offsets.append(out_f.tell())
            rec_bytes = (json.dumps(record, ensure_ascii=False) + '\n').encode('utf-8')
            out_f.write(rec_bytes)

    index_path = output_path.with_suffix('.idx.json')
    with index_path.open('w', encoding='utf-8') as idx_f:
        json.dump(
            {
                'source_file': output_path.name,
                'byte_offsets': byte_offsets,
            },
            idx_f,
            ensure_ascii=False,
            indent=2,
        )


for file_idx in range(1000):
    file_path = INSTRUCTION_ROOT / f"training_tfexample.tfrecord-{file_idx:05d}-of-01000_t10.jsonl"
    with open(file_path, 'r', encoding='utf-8') as f:
        records = []
        scenario_index = 0
        for i, item in tqdm(enumerate(f), desc=f"File {file_idx:05d}"):
            record = json.loads(item)
            annot = record['annotation']['ego_motion']
            if scenario_index != record['scenario_index']:
                # raise ValueError(f"Scenario index mismatch: {scenario_index} vs {record['scenario_index']}")
                continue
            # speed instruction
            if float(annot['start_speed']) < 0.1:
                if float(annot['end_speed']) < 0.1:
                    speed_inst = 'stop'
                elif float(annot['end_speed']) > 0.1:
                    speed_inst = 'stop and go'
            elif np.abs(float(annot['start_speed']) - float(annot['end_speed'])) / float(annot['start_speed']) < 0.1:
                speed_inst = 'maintain'
            elif float(annot['start_speed']) < float(annot['end_speed']):
                speed_inst = 'accelerate'
            else:
                speed_inst = 'decelerate'
            lane_is_known = True
            if speed_inst == 'stop':
                instruction = 'stop'
            else:
                instruction = ''
                if speed_inst == 'stop and go':
                    instruction += 'stop for a while, and then '

                if annot['direction'] == 'straight':
                    instruction += 'go straight '
                elif annot['direction'] == 'left':
                    instruction += 'go left '
                elif annot['direction'] == 'right':
                    instruction += 'go right '
                elif annot['direction'] == 'slight left':
                    instruction += 'go slightly left '
                elif annot['direction'] == 'slight right':
                    instruction += 'go slightly right '
                else:
                    instruction += 'go '

                if 'left' in annot['turn_label']:
                    instruction += 'to turn left '
                elif 'right' in annot['turn_label']:
                    instruction += 'to turn right '
                elif 'u-turn' in annot['turn_label']:
                    instruction += 'to make a U-turn '
                
                if 'left' in annot['lane_change']:
                    instruction += 'while changing lane to the left '
                elif 'right' in annot['lane_change']:
                    instruction += 'while changing lane to the right '
                elif 'none' in annot['lane_change']:
                    instruction += 'while following current lane '
                else:
                    lane_is_known = False
                
                if speed_inst == 'accelerate':
                    instruction += 'and accelerating' if lane_is_known else 'while accelerating'
                elif speed_inst == 'decelerate':
                    instruction += 'and slowing down' if lane_is_known else 'while slowing down'
            record['instruction'] = instruction
            records.append(record)

            scenario_index += 1

        output_path = TARGET_ROOT / f"training_tfexample.tfrecord-{file_idx:05d}-of-01000_t10.jsonl"
        write_jsonl_with_byte_index(records, output_path)


File 00000: 455it [00:00, 73428.56it/s]
File 00001: 0it [00:00, ?it/s]

File 00001: 479it [00:00, 54835.73it/s]
File 00002: 514it [00:00, 49906.76it/s]
File 00003: 479it [00:00, 40317.71it/s]
File 00004: 495it [00:00, 69236.02it/s]
File 00005: 465it [00:00, 44370.54it/s]
File 00006: 516it [00:00, 47841.66it/s]
File 00007: 468it [00:00, 80580.22it/s]
File 00008: 499it [00:00, 65398.80it/s]
File 00009: 481it [00:00, 41041.99it/s]
File 00010: 476it [00:00, 47661.41it/s]
File 00011: 501it [00:00, 62260.27it/s]
File 00012: 509it [00:00, 15327.90it/s]
File 00013: 476it [00:00, 46961.84it/s]
File 00014: 484it [00:00, 55194.21it/s]
File 00015: 494it [00:00, 55942.17it/s]
File 00016: 468it [00:00, 53567.69it/s]
File 00017: 453it [00:00, 60178.62it/s]
File 00018: 465it [00:00, 44561.13it/s]
File 00019: 487it [00:00, 58677.60it/s]
File 00020: 476it [00:00, 61824.19it/s]
File 00021: 450it [00:00, 49728.28it/s]
File 00022: 469it [00:00, 41852.02it/s]
File 00023: 478it [00:00, 45013.97it/s]
File 00024: 463it [00:00, 39732.44it/s]
File 00025: 499it [00:00, 57355.45it/s]


{
  "tfrecord_path": "training_tfexample.tfrecord-00999-of-01000",
  "scenario_index": 0,
  "annotation": {
    "scenario_window": {
      "start": 10,
      "end": 60
    },
    "ego_motion": {
      "start_xy": "[  450.1722 -1067.4644]",
      "end_xy": "[  450.21756 -1129.9393 ]",
      "start_yaw": "-1.5700696",
      "end_yaw": "-1.5732796",
      "start_speed": "13.399072",
      "end_speed": "11.815027",
      "avg_speed": "12.747044",
      "direction": "straight",
      "turn_label": "straight",
      "lane_change": "none"
    },
    "lane_context": {
      "start_lane_id": 182,
      "end_lane_id": 215,
      "left_lane_ids": [
        180,
        194,
        197
      ],
      "right_lane_ids": [],
      "traffic_lights": [
        "no traffic light"
      ]
    },
    "risk_objects": {
      "front": [],
      "left": [
        {
          "object_index": 2,
          "object_type": "vehicle",
          "lateral_distance": 3.513415813446045,
          "longitudinal_distan